# Task 1 — Experimental Protocol & Baseline Recommenders

Notebook ini dibuat berdasarkan ketentuan Task 1 yang telah dibagikan:
- Gunakan **official temporal split**; jangan melakukan random shuffle atau membuat split baru.
- Ukuran acuan: **Training = 427.957**, **Validation = 41.284**, **Test = 42.595**.
- Gunakan aturan **warm-start evaluable users** dari paket proyek.
- Evaluasi utama: **Top-10**, dengan **Recall@10** dan **NDCG@10**.
- Test hanya untuk evaluasi final pada W13; jangan digunakan untuk tuning/model selection.

Jalankan notebook dari folder utama proyek yang berisi `src/`, `MAIN DATA-...`, dan `SUPPORTING_DATA-...`.

**Catatan:** Paket proyek yang tampak pada notebook sebelumnya menyediakan validation warm, evaluable users warm, dan candidate universe warm. Notebook ini mengaudit official split dan menjalankan baseline pada protokol warm-start yang disediakan. Cocokkan kembali pemilihan file evaluasi dengan instruksi lengkap Task 1 jika ada ketentuan tambahan.


## 1. Project setup dan import

Task ini bertujuan mempersiapkan protokol eksperimen dan membangun model baseline rekomendasi untuk menghasilkan serta mengevaluasi rekomendasi Top-10 menggunakan Recall@10 dan NDCG@10.

In [3]:
from pathlib import Path
import sys
import pandas as pd
import numpy as np
from scipy import sparse
from sklearn.feature_extraction.text import TfidfVectorizer

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "src").is_dir():
    raise FileNotFoundError("Jalankan notebook dari folder utama proyek yang berisi src/.")

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# Hanya import fungsi yang memang tersedia di src/data.py.
# Audit warm-start dilakukan langsung di notebook ini, tanpa mengubah folder src/.
from src.data import load_project_data
from src.evaluation import evaluate_ranking
from src.recommenders import (
    fit_popularity, fit_item_cf, fit_tfidf_cbf,
    recommend_popularity, recommend_from_model, _prepare_matrices
)

K = 10
TOP_N = 10
POSITIVE_THRESHOLD = 4
print("Project root:", PROJECT_ROOT)


Project root: d:\Hybrid-Fusion-Strategies-for-Collaborative-and-Content-Based-Recommender-Systems\Hybrid-Fusion-Strategies-for-Collaborative-and-Content-Based-Recommender-Systems


## 2. Audit official temporal split\nTidak ada random split atau `train_test_split`. Data dibaca dari file temporal split yang disediakan.

Memastikan pembagian data menggunakan official temporal split sesuai ketentuan, tanpa melakukan random split atau pengacakan ulang data.

In [4]:
main_dirs = list(PROJECT_ROOT.glob("MAIN DATA-*/MAIN DATA"))
if len(main_dirs) != 1:
    raise FileNotFoundError(f"Ditemukan {len(main_dirs)} folder MAIN DATA; periksa struktur dataset.")
MAIN_DIR = main_dirs[0]

train_official = pd.read_csv(MAIN_DIR / "interactions_train.csv",
    dtype={"user_id": "string", "item_id": "string"})
validation_official = pd.read_csv(MAIN_DIR / "interactions_validation.csv",
    dtype={"user_id": "string", "item_id": "string"})

test_paths = [MAIN_DIR / "interactions_test.csv", MAIN_DIR / "interactions_test_final.csv"]
TEST_PATH = next((p for p in test_paths if p.exists()), None)
test_official = None
if TEST_PATH is not None:
    test_official = pd.read_csv(TEST_PATH,
        dtype={"user_id": "string", "item_id": "string"})
else:
    print("File test tidak ditemukan dengan nama standar. Periksa nama file test di MAIN DATA.")

print("Training :", len(train_official), "(target 427,957)")
print("Validation:", len(validation_official), "(target 41,284)")
if test_official is not None:
    print("Test     :", len(test_official), "(target 42,595)")

assert len(train_official) == 427_957, "Ukuran training tidak sesuai acuan."
assert len(validation_official) == 41_284, "Ukuran validation tidak sesuai acuan."
if test_official is not None:
    assert len(test_official) == 42_595, "Ukuran test tidak sesuai acuan."


File test tidak ditemukan dengan nama standar. Periksa nama file test di MAIN DATA.
Training : 427957 (target 427,957)
Validation: 41284 (target 41,284)


## 3. Muat protokol warm-start proyek
`load_project_data()` mengambil training, metadata item, candidate universe warm, validation warm, dan daftar evaluable users warm dari file pendukung yang sudah disediakan. Tidak membuat split baru.


In [5]:
data = load_project_data(PROJECT_ROOT)

train = data.train.copy()
items = data.items.copy()
candidates = list(dict.fromkeys(data.candidates))
eval_users = list(dict.fromkeys(data.evaluable_users))
validation_warm = data.validation_warm.copy()

# Samakan tipe ID agar proses filtering tidak gagal karena perbedaan tipe data.
train["user_id"] = train["user_id"].astype(str)
train["item_id"] = train["item_id"].astype(str)
items["item_id"] = items["item_id"].astype(str)
validation_warm["user_id"] = validation_warm["user_id"].astype(str)
validation_warm["item_id"] = validation_warm["item_id"].astype(str)
candidates = list(dict.fromkeys(map(str, candidates)))
eval_users = list(dict.fromkeys(map(str, eval_users)))

print(f"Training interactions: {len(train):,}")
print(f"Training users: {train.user_id.nunique():,}")
print(f"Training items: {train.item_id.nunique():,}")
print(f"Warm candidates: {len(candidates):,}")
print(f"Evaluable users: {len(eval_users):,}")
print(f"Warm validation rows: {len(validation_warm):,}")


Training interactions: 427,957
Training users: 54,483
Training items: 22,753
Warm candidates: 22,753
Evaluable users: 11,659
Warm validation rows: 27,537


## 4. Evaluable-user filtering dan pembentukan target relevan
Gunakan hanya user yang tercantum pada `evaluable_users_validation_warm.csv`. Target relevan berasal dari `interactions_validation_warm.csv`, memiliki rating minimal 4, dan item-nya harus termasuk candidate universe. Semua langkah filtering dilakukan di notebook Task 1 ini.


In [6]:
train_user_set = set(train["user_id"])
eval_user_set = set(eval_users)
candidate_set = set(candidates)

# 1) Semua evaluable users harus memiliki riwayat pada data training (warm-start).
missing_users = eval_user_set - train_user_set
assert not missing_users, (
    f"Ada {len(missing_users):,} evaluable users yang tidak ditemukan di training."
)

# 2) Validation hanya memuat evaluable users.
validation_warm = validation_warm[
    validation_warm["user_id"].isin(eval_user_set)
].copy()

# 3) Target relevan dibatasi pada item kandidat agar denominator evaluasi
#    tidak menghitung item yang memang tidak boleh direkomendasikan.
relevant_rows = validation_warm[
    (validation_warm["rating"] >= POSITIVE_THRESHOLD)
    & (validation_warm["item_id"].isin(candidate_set))
].copy()
relevant = (
    relevant_rows.groupby("user_id")["item_id"]
    .apply(set)
    .to_dict()
)

print("Evaluable users dari file protokol:", len(eval_user_set))
print("Evaluable users yang punya riwayat training:", len(eval_user_set - missing_users))
print("Validation warm rows setelah filter user:", len(validation_warm))
print("Validation target relevan setelah filter rating + kandidat:", len(relevant_rows))
print("Users dengan minimal satu target relevan:", len(relevant))

assert set(relevant.keys()).issubset(eval_user_set)
assert all(targets.issubset(candidate_set) for targets in relevant.values())


Evaluable users dari file protokol: 11659
Evaluable users yang punya riwayat training: 11659
Validation warm rows setelah filter user: 27537
Validation target relevan setelah filter rating + kandidat: 24202
Users dengan minimal satu target relevan: 11659


## 5. Candidate-universe filtering
Candidate universe dibaca dari file resmi `candidate_items_warm.csv`. Ketiga baseline menerima daftar kandidat yang sama. Target validation juga dibatasi pada candidate universe agar evaluasi Recall@10/NDCG@10 sejalan dengan ruang item yang boleh direkomendasikan.


In [7]:
train_item_set = set(train["item_id"])
item_catalog_set = set(items["item_id"])

print("Candidate universe:", len(candidate_set))
print("Candidates yang ada di metadata item:", len(candidate_set & item_catalog_set))
print("Candidates yang juga muncul di training:", len(candidate_set & train_item_set))

assert candidate_set, "Candidate universe kosong."
assert candidate_set.issubset(item_catalog_set), (
    "Ada candidate item yang tidak ditemukan di metadata items.csv."
)


Candidate universe: 22753
Candidates yang ada di metadata item: 22753
Candidates yang juga muncul di training: 22753


## 6. Baseline 1 — Popularity\nPopularity menghitung jumlah interaksi positif (rating ≥ 4) pada training dan menjadi baseline tanpa personalisasi.

Menghitung popularitas item berdasarkan jumlah interaksi positif (rating ≥ 4) pada data training untuk menghasilkan rekomendasi tanpa personalisasi.

In [8]:
popularity = fit_popularity(train, positive_threshold=POSITIVE_THRESHOLD)

seen_by_user = {
    str(user): set(group["item_id"].astype(str))
    for user, group in train.groupby("user_id", sort=False)
}

pop_recs = recommend_popularity(
    eval_users, candidates, popularity, seen_by_user, top_n=TOP_N
)
pop_metrics = evaluate_ranking(pop_recs, relevant, eval_users, k=K)
pop_metrics


{'users_evaluated': 11659,
 'recall@10': 0.01349850373292779,
 'ndcg@10': 0.009222040875545047}

## 7. Baseline 2 — Item-Based Collaborative Filtering (Item-CF)\nItem-CF menghitung kemiripan item berdasarkan interaksi positif bersama pada data training.

Mengukur kemiripan antar-item berdasarkan pola interaksi positif pengguna pada data training untuk menghasilkan rekomendasi.

In [9]:
item_cf_model = fit_item_cf(
    train, candidates, top_k=100, positive_threshold=POSITIVE_THRESHOLD
)
item_cf_recs = recommend_from_model(
    item_cf_model, eval_users, candidates, popularity, top_n=TOP_N
)
item_cf_metrics = evaluate_ranking(item_cf_recs, relevant, eval_users, k=K)
item_cf_metrics


d:\Anacondaa\Lib\site-packages\scipy\sparse\_index.py:146: SparseEfficiencyWarning: Changing the sparsity structure of a csr_matrix is expensive. lil_matrix is more efficient.
  self._set_arrayXarray(i, j, x)


{'users_evaluated': 11659,
 'recall@10': 0.005799512862838689,
 'ndcg@10': 0.003323031558715048}

## 8. Baseline 3 — TF-IDF Content-Based Filtering (CBF)\nTF-IDF membentuk representasi teks dari kolom `content_text_basic`, menghitung kemiripan antar-item, lalu menggunakan riwayat positif user.

**Catatan memori:** perhitungan kemiripan TF-IDF dilakukan per blok di notebook ini (tanpa mengubah `src/`) agar tidak mengalokasikan seluruh matriks item-item sekaligus. Jika RAM masih penuh, ubah `CHUNK_SIZE = 32` menjadi `16` atau `8`.


Pada tahap ini, sistem menggunakan informasi teks item untuk menghitung kemiripan dan menghasilkan rekomendasi berdasarkan riwayat interaksi positif pengguna.

In [10]:
assert "content_text_basic" in items.columns, (
    "Kolom content_text_basic tidak ditemukan. Periksa nama kolom pada dataset/modul."
)

# Bangun representasi TF-IDF seperti implementasi src, tetapi hitung kemiripan
# per blok agar tidak membuat matriks item-item besar sekaligus di RAM.
user_ids, item_ids, user_index, item_index, interactions, seen = _prepare_matrices(
    train, candidates, POSITIVE_THRESHOLD
)
metadata = items.copy()
metadata["item_id"] = metadata["item_id"].astype(str)
text_by_id = metadata.set_index("item_id")["content_text_basic"].fillna("").astype(str)
documents = [text_by_id.get(item_id, "") for item_id in item_ids]

vectorizer = TfidfVectorizer(
    lowercase=True,
    stop_words="english",
    ngram_range=(1, 2),
    min_df=2,
    max_df=0.8,
    max_features=100_000,
    sublinear_tf=True,
    dtype=np.float32,
)
item_vectors = vectorizer.fit_transform(documents).tocsr()

# Top-100 tetangga per item. Ubah CHUNK_SIZE menjadi 16 jika RAM masih terbatas.
TOP_K_SIMILARITY = 100
CHUNK_SIZE = 32
n_items = item_vectors.shape[0]
rows, cols, values = [], [], []

for start in range(0, n_items, CHUNK_SIZE):
    stop = min(start + CHUNK_SIZE, n_items)
    block_similarity = (item_vectors[start:stop] @ item_vectors.T).tocsr()
    block_similarity.eliminate_zeros()

    for local_row in range(stop - start):
        global_row = start + local_row
        row_start = block_similarity.indptr[local_row]
        row_end = block_similarity.indptr[local_row + 1]
        row_cols = block_similarity.indices[row_start:row_end]
        row_values = block_similarity.data[row_start:row_end]

        # Jangan masukkan item itu sendiri sebagai tetangga.
        valid = row_cols != global_row
        row_cols, row_values = row_cols[valid], row_values[valid]
        if len(row_values) > TOP_K_SIMILARITY:
            keep = np.argpartition(row_values, -TOP_K_SIMILARITY)[-TOP_K_SIMILARITY:]
            row_cols, row_values = row_cols[keep], row_values[keep]

        rows.extend([global_row] * len(row_cols))
        cols.extend(row_cols.tolist())
        values.extend(row_values.astype(np.float32).tolist())

    del block_similarity

similarity_graph = sparse.csr_matrix(
    (np.asarray(values, dtype=np.float32),
     (np.asarray(rows, dtype=np.int32), np.asarray(cols, dtype=np.int32))),
    shape=(n_items, n_items),
    dtype=np.float32,
)

tfidf_cbf_model = {
    "user_ids": user_ids,
    "item_ids": item_ids,
    "user_index": user_index,
    "item_index": item_index,
    "interactions": interactions,
    "seen": seen,
    "similarity": similarity_graph,
    "vectorizer": vectorizer,
}

tfidf_cbf_recs = recommend_from_model(
    tfidf_cbf_model, eval_users, candidates, popularity, top_n=TOP_N
)
tfidf_cbf_metrics = evaluate_ranking(tfidf_cbf_recs, relevant, eval_users, k=K)
tfidf_cbf_metrics


{'users_evaluated': 11659,
 'recall@10': 0.012148205722961746,
 'ndcg@10': 0.007490868620646675}

## 9. Consistency checks — evaluable users, candidates, dan Top-10

Tahap ini dilakukan untuk memastikan seluruh model menghasilkan rekomendasi yang konsisten, sesuai kandidat item, dan tidak merekomendasikan item yang sudah pernah dilihat pengguna.

In [11]:
all_recommendations = {
    "Popularity": pop_recs,
    "Item-CF": item_cf_recs,
    "TF-IDF CBF": tfidf_cbf_recs,
}

for model_name, recs in all_recommendations.items():
    assert set(recs.keys()) == eval_user_set, (
        f"{model_name}: daftar user hasil rekomendasi tidak sama."
    )

    candidate_violations = []
    seen_violations = []
    topn_violations = []

    for user, ranked in recs.items():
        ids = [str(item_id) for item_id, _ in ranked]
        if not set(ids).issubset(candidate_set):
            candidate_violations.append(user)
        if any(item_id in seen_by_user.get(str(user), set()) for item_id in ids):
            seen_violations.append(user)
        if len(ids) > K:
            topn_violations.append(user)

    print(
        model_name,
        "| users:", len(recs),
        "| candidate violations:", len(candidate_violations),
        "| seen-item violations:", len(seen_violations),
        "| >Top-10:", len(topn_violations)
    )
    assert not candidate_violations
    assert not seen_violations
    assert not topn_violations

print("Consistency checks: PASSED")


Popularity | users: 11659 | candidate violations: 0 | seen-item violations: 0 | >Top-10: 0
Item-CF | users: 11659 | candidate violations: 0 | seen-item violations: 0 | >Top-10: 0
TF-IDF CBF | users: 11659 | candidate violations: 0 | seen-item violations: 0 | >Top-10: 0
Consistency checks: PASSED


## 10. Perbandingan metrik utama — Recall@10 dan NDCG@10

Pada tahap ini, hasil evaluasi ketiga model dibandingkan menggunakan Recall@10 dan NDCG@10 untuk mengetahui performa masing-masing model.

In [12]:
results = pd.DataFrame([
    {"model": "Popularity", **pop_metrics},
    {"model": "Item-CF", **item_cf_metrics},
    {"model": "TF-IDF CBF", **tfidf_cbf_metrics},
]).set_index("model")

results


,users_evaluated,recall@10,ndcg@10
model,,,
Popularity,11659,0.013499,0.009222
Item-CF,11659,0.005800,0.003323
TF-IDF CBF,11659,0.012148,0.007491


## 11. Contoh hasil rekomendasi satu user (inspeksi)

Tahap ini menampilkan contoh hasil rekomendasi dari masing-masing model kepada satu pengguna sebagai bentuk inspeksi hasil rekomendasi.

In [13]:
sample_user = eval_users[0]

def show_recommendations(recommendation_dict, user_id, model_name):
    rec_df = pd.DataFrame(
        recommendation_dict.get(str(user_id), []),
        columns=["item_id", "score"]
    )
    rec_df["item_id"] = rec_df["item_id"].astype(str)

    if "title" in items.columns:
        item_meta = items.copy()
        item_meta["item_id"] = item_meta["item_id"].astype(str)
        rec_df = rec_df.merge(
            item_meta[["item_id", "title"]].drop_duplicates("item_id"),
            on="item_id", how="left"
        )
        display(rec_df[["item_id", "title", "score"]].head(K))
    else:
        display(rec_df.head(K))

    print(f"{model_name} — user {user_id}")

for model_name, recs in all_recommendations.items():
    show_recommendations(recs, sample_user, model_name)


,item_id,title,score
0,B09857JRP2,GLS Audio Instrument Cable - Amp Cord for Bass...,2175.0
1,B0BCK6L7S5,"Donner Guitar Delay Pedal, Yellow Fall Analog ...",1441.0
2,B0BSGM6CQ9,Ernie Ball Mondo Slinky Nickelwound Electric G...,1404.0
3,B0BPJ4Q6FJ,D'Addario Guitar Strings - XL Nickel Electric ...,1271.0
4,B0BTC9YJ2W,D'Addario Guitar Strings - Phosphor Bronze Aco...,1185.0
5,B004XNK7AI,"Snark SN-5 Tuner for Guitar, Bass and Violin (...",1119.0
6,B0BHSG5C3G,"Snark SN5X Clip-On Tuner for Guitar, Bass & Vi...",1031.0
7,B08R5GM6YB,"D'Addario Pro-Winder - Guitar String Winder, G...",1002.0
8,B0C67HCGBR,GLS Audio 12ft Mic Cable Patch Cords - XLR Mal...,917.0
9,B0BPLFP5P6,D'Addario Accessories Guitar Tuner - Micro Hea...,895.0


Popularity — user AE22B7BMXT7GAGY3XLHICMESNAPA


,item_id,title,score
0,B09XXG98XF,"Korg, Metronome (KDM3WH)",0.500000
1,B07JLFRBN8,Alnicov Set of 3 Alnico V Single-Coil-Sized Hu...,0.288675
2,B08Y1NRCWW,Gator Frameworks Utility Table Top and X Style...,0.277350
3,B08BQJYNCQ,Premium Guitar Spanner Wrench with thin Foam s...,0.250000
4,B08GS84HXJ,Donner DPA-100 Guitar Pedal Power Supply Adapt...,0.235702
5,B01L1GCETG,Vizcaya Black Rubber Clarinet Thumb Rest Cushi...,0.235702
6,B00HYMBMEU,Aquila 107U Tenor Ukulele Strings Set,0.235702
7,B0BZ5BMQ5P,"Buckle-Down DC Comics Guitar Strap, Superman S...",0.223607
8,B00LA10IUU,Neweights Cotton Rainbow Color Ukulele Strap w...,0.213201
9,B0C2PY3PJV,"JYX Portable Bluetooth Karaoke Machine, Microp...",0.208514


Item-CF — user AE22B7BMXT7GAGY3XLHICMESNAPA


,item_id,title,score
0,B086QKG5WZ,Boss MD-200 Modulation Pedal,0.650861
1,B07T9FHGWG,Boss EQ-200 Graphic Equalizer Pedal,0.574329
2,B00KAVMRXW,NUX Time Force Multi Modulation Digital Delay ...,0.501308
3,B07H75VQ5R,NUX Monterey Vibe Guitar Effects Pedal with an...,0.495820
4,B00HI4CULS,"Audio2000's C27003P2 3 Feet 1/4"" TS Right Angl...",0.493052
5,B00BK3VBWC,Nux Time Core Delay Guitar Effects,0.480852
6,B01H9DB18K,"Audio2000's C27012P1 12 Ft 1/4"" TS Right Angle...",0.466915
7,B00SXWIG8A,"Audio 2000s E28103P1 1/4"" TS Right Angle to 1/...",0.466875
8,B07DK964NQ,NUX Loop Core Deluxe 24-Bit Looper Pedal,0.465400
9,B07BRG8FF2,Audio2000'S ADC203MX3 3-Pack XLR Male to XLR F...,0.461185


TF-IDF CBF — user AE22B7BMXT7GAGY3XLHICMESNAPA
